In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/StoichModelGP/ModelGP_M25.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7695450464589964, 'n_it': 0.39672112416366373}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 300

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.PseudorandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[13.647934491548739, 14.804245837811798, 15.6773836242021, 13.791450485568538, 14.81082915359157, 13.600648568847912, 14.86246233141481, 18.245617470441612, 14.01686172441376, 14.643633863648251, 14.199988882112068, 13.364429457158963, 14.507532015498972, 13.44169906342425, 13.503310213272421, 14.948916375396252, 16.129558904300715, 14.516434861976787, 13.506311034340394, 16.20979729723635, 13.142244085028079, 13.347715960895517, 16.899930667081133, 13.70331304527061, 17.120223788806378, 13.724665055034682, 13.706873714793176, 16.279049428133025, 13.378086068539755, 13.854849617993072, 14.964253525846862, 13.595135991370228, 13.726892387765677, 13.289591095217874, 15.991489815812464, 15.832084102563982, 16.74425371610698, 13.300827605160618, 14.268590052614865, 13.682799915929184, 17.370130607145676, 14.224008589950111, 16.60423941815212, 13.460326545684012, 13.872244445393028, 14.405346094288713, 13.338762312063926, 15.832793057140364, 14.08494978861307, 17.043741101517014, 15.7490481

In [5]:
np.average(y_max_arr)

np.float64(14.96178260608645)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_M25/DataGenerated/pseudorandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)